# Venture Corpus C1 — quality audit

## TL;DR

C1 contains 2 calibration and 8 evaluation cases. Numeric graph inputs contain no
text, all sources are later than C0, and organization/CIK/accession overlap is zero.
Candidate generation remains blocked until an independent reviewer verifies the
source-to-graph annotations.

## Context and methods

The intended grain is one organization filing, one registered business challenge
and one typed numeric graph. This notebook reads the committed manifest, safe NPZ
archive, audit sidecar and generated quality report. It does not contact external
services and does not create experiment outcomes.

In [1]:
import json
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
DATA = ROOT / "datasets" / "venture_corpus_c1"
manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
quality = json.loads((DATA / "quality_report.json").read_text(encoding="utf-8"))
case_lines = (DATA / "cases.jsonl").read_text(encoding="utf-8").splitlines()
cases = [json.loads(line) for line in case_lines]
with np.load(DATA / "graphs.npz", allow_pickle=False) as archive:
    arrays = {name: archive[name].copy() for name in archive.files}

print({
    "corpus_id": manifest["corpus_id"],
    "cases": manifest["counts"]["cases"],
    "calibration": manifest["counts"]["calibration"],
    "evaluation": manifest["counts"]["evaluation"],
    "release_status": manifest["release_status"],
})

{'corpus_id': 'CHM-VENTURE-C1', 'cases': 10, 'calibration': 2, 'evaluation': 8, 'release_status': 'provisional_pending_independent_annotation_review'}


## Data

In [2]:
node_counts = arrays["graph_node_mask"].sum(axis=1)
edge_counts = (arrays["graph_edge_types"] > 0).sum(axis=(1, 2))
print(f"{'case_id':28} {'partition':12} {'period':10} {'nodes':>5} {'edges':>5}")
for record, nodes, edges in zip(cases, node_counts, edge_counts, strict=True):
    print(
        f"{record['case_id']:28} {record['partition']:12} "
        f"{record['source']['period_end']:10} {int(nodes):5d} {int(edges):5d}"
    )

case_id                      partition    period     nodes edges
sec-walmart-2025             calibration  2025-01-31    13     9
sec-salesforce-2025          calibration  2025-01-31    11     9
sec-microsoft-2025           evaluation   2025-06-30    13    11
sec-nvidia-2025              evaluation   2025-01-26    14    12
sec-home-depot-2025          evaluation   2025-02-02    12     9
sec-intuit-2025              evaluation   2025-07-31    12    10
sec-fedex-2025               evaluation   2025-05-31    12     9
sec-nike-2025                evaluation   2025-05-31    14    12
sec-oracle-2025              evaluation   2025-05-31    13    11
sec-pg-2025                  evaluation   2025-06-30    12     8


## Results

In [3]:
dimensions = quality["dimensions"]
summary = {
    "unique_case_ids": dimensions["uniqueness"]["unique_case_ids"],
    "unique_numeric_graphs": dimensions["uniqueness"]["unique_numeric_graphs"],
    "unique_topologies": dimensions["uniqueness"]["unique_topologies"],
    "feature_range": [
        dimensions["validity"]["feature_min"],
        dimensions["validity"]["feature_max"],
    ],
    "finite_features": dimensions["validity"]["finite_features"],
    "numeric_archive_has_text_or_objects": dimensions["validity"][
        "numeric_archive_has_text_or_objects"
    ],
    "case_aligned_briefs": dimensions["consistency"]["case_aligned_briefs"],
}
print(json.dumps(summary, indent=2))

{
  "unique_case_ids": 10,
  "unique_numeric_graphs": 10,
  "unique_topologies": 10,
  "feature_range": [
    0.0,
    1.0
  ],
  "finite_features": true,
  "numeric_archive_has_text_or_objects": false,
  "case_aligned_briefs": true
}


In [4]:
boundary = manifest["temporal_boundary"]
leakage = quality["dimensions"]["leakage"]
print(json.dumps({
    "pretraining_max_period_end": boundary["pretraining_max_period_end"],
    "evaluation_min_period_end": boundary["evaluation_min_period_end"],
    "organization_overlap_count": leakage["organization_overlap_count"],
    "cik_overlap_count": leakage["cik_overlap_count"],
    "accession_overlap_count": leakage["accession_overlap_count"],
    "outcome_labels_present": leakage["outcome_labels_present"],
}, indent=2))

{
  "pretraining_max_period_end": "2024-12-31",
  "evaluation_min_period_end": "2025-01-26",
  "organization_overlap_count": 0,
  "cik_overlap_count": 0,
  "accession_overlap_count": 0,
  "outcome_labels_present": false
}


In [5]:
source_review = quality["dimensions"]["source_review"]
gate = source_review["gate"]
print(json.dumps({
    "internal_filing_identity_verified": source_review[
        "internal_filing_identity_verified"
    ],
    "internal_primary_source_support_verified": source_review[
        "internal_primary_source_support_verified"
    ],
    "internal_auditor_independent": source_review[
        "internal_auditor_independent"
    ],
    "independent_reviews": (
        f"{gate['accepted_reviews']}/{gate['minimum_independent_reviewers']}"
    ),
    "review_gate_status": gate["status"],
    "generation_allowed": gate["generation_allowed"],
}, indent=2))

{
  "internal_filing_identity_verified": 10,
  "internal_primary_source_support_verified": 10,
  "internal_auditor_independent": false,
  "independent_reviews": "0/1",
  "review_gate_status": "blocked",
  "generation_allowed": false
}


In [6]:
for finding in quality["findings"]:
    print(
        f"[{finding['severity'].upper()}] {finding['id']} — "
        f"{finding['status']}: {finding['finding']}"
    )
print()
print("Fitness:")
print(json.dumps(quality["fitness_for_use"], indent=2))

[HIGH] C1-Q001 — open: Internal primary-source checks cover all ten cases; independent reviews accepted: 0/1.
[HIGH] C1-Q002 — accepted_limitation: C1 contains public-company SEC filings only.
[HIGH] C1-Q003 — expected_by_design: The corpus contains no creativity, feasibility or commercial outcome labels.
[MEDIUM] C1-Q004 — accepted_limitation: The primary analysis has eight independent evaluation cases.

Fitness:
{
  "candidate_generation": "blocked_pending_C1-Q001",
  "corpus_build_and_protocol_preregistration": "fit",
  "creativity_claim": "not_fit_until_blind_ratings_and_locked_analysis"
}


## Takeaways

- The numeric contract, temporal isolation and matched baseline alignment pass.
- Internal primary-source checks cover 10/10 cases; independent review is 0/1.
- C1 is fit for preregistration, not yet for candidate generation.
- A second reviewer must close `C1-Q001` before the frozen H001 run.
- H001 remains `not_run`; this notebook contains no creativity result.